# Prodigal + ESM-2

In [ ]:
# Install prodigal
!apt-get update && apt-get install prodigal -y

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,806 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/u

In [ ]:
!pip install streamlit
!pip install esm
!pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 132.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.0/58.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.

In [ ]:
# Install fair-esm and torch
!pip install fair-esm torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 3.4 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st
import os
import subprocess
import torch
import esm
import numpy as np
from Bio import SeqIO
import io

st.set_page_config(page_title="Genome to ESM-2", layout="wide")
st.title("Genome to ESM-2 Embeddings")
st.markdown("Upload a genome file (`.fna` or `.fasta`) to run **Prodigal** and generate **ESM-2** protein embeddings.")

# Initialize session state for persistence
if 'proteins_ready' not in st.session_state:
    st.session_state.proteins_ready = False
if 'embeddings_ready' not in st.session_state:
    st.session_state.embeddings_ready = False
if 'final_embeddings' not in st.session_state:
    st.session_state.final_embeddings = None

# Load Model once
@st.cache_resource
def load_esm_model():
    model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    return model, alphabet, device

model, alphabet, device = load_esm_model()
batch_converter = alphabet.get_batch_converter()

uploaded_file = st.file_uploader("Choose a genome file", type=["fna", "fasta"])

if uploaded_file is not None:
    # Reset if a new file is uploaded
    if "last_uploaded" not in st.session_state or st.session_state.last_uploaded != uploaded_file.name:
        st.session_state.proteins_ready = False
        st.session_state.embeddings_ready = False
        st.session_state.last_uploaded = uploaded_file.name

    with open("temp_genome.fna", "wb") as f:
        f.write(uploaded_file.getbuffer())

    if not st.session_state.proteins_ready:
        st.info("Running Prodigal...")
        try:
            subprocess.run([
                "prodigal", "-i", "temp_genome.fna",
                "-a", "temp_proteins.faa",
                "-p", "single"
            ], check=True, capture_output=True)
            st.session_state.proteins_ready = True
            st.success("Prodigal finished!")
        except Exception as e:
            st.error(f"Prodigal error: {e}")

    if st.session_state.proteins_ready:
        records = list(SeqIO.parse("temp_proteins.faa", "fasta"))
        st.write(f"Found {len(records)} proteins.")

        if st.button("Generate ESM-2 Embeddings") or st.session_state.embeddings_ready:
            if not st.session_state.embeddings_ready:
                progress_bar = st.progress(0)
                embeddings_list = []

                for i, record in enumerate(records):
                    seq = str(record.seq).replace('*', '')[:1024]
                    data = [("protein", seq)]
                    _, _, batch_tokens = batch_converter(data)
                    batch_tokens = batch_tokens.to(device)

                    with torch.no_grad():
                        results = model(batch_tokens, repr_layers=[33])
                        token_reps = results["representations"][33]
                        mean_rep = token_reps[0, 1 : len(seq) + 1].mean(0).cpu().numpy()
                        embeddings_list.append(mean_rep)

                    progress_bar.progress((i + 1) / len(records))

                st.session_state.final_embeddings = np.array(embeddings_list)
                st.session_state.embeddings_ready = True
                st.rerun()

            if st.session_state.embeddings_ready:
                st.success(f"Embeddings ready: {st.session_state.final_embeddings.shape}")

                with open("temp_proteins.faa", "rb") as f:
                    st.download_button("Download Proteins (.faa)", data=f, file_name="predicted_proteins.faa")

                buffer = io.BytesIO()
                np.save(buffer, st.session_state.final_embeddings)
                st.download_button("Download Embeddings (.npy)", data=buffer.getvalue(), file_name="embeddings.npy")

Writing app.py


In [ ]:
import subprocess
import os
import time

# 1. Download and install bore binary
if not os.path.exists('bore'):
    print("Installing bore...")
    !wget -q https://github.com/ekzhang/bore/releases/download/v0.5.1/bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
    !tar -xzf bore-v0.5.1-x86_64-unknown-linux-musl.tar.gz
    !chmod +x bore

# 2. Start Streamlit in the background
print("Starting Streamlit...")
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0'])
time.sleep(5)

# 3. Run bore to expose the port
print("\nConnecting to bore.pub...")
print("Your app will be available at: bore.pub:[REMOTE_PORT]")
!./bore local 8501 --to bore.pub

Installing bore...
Starting Streamlit...

Connecting to bore.pub...
Your app will be available at: bore.pub:[REMOTE_PORT]
2026-07-06T08:51:59.128892Z  INFO bore_cli::client: connected to server remote_port=61081
2026-07-06T08:51:59.128936Z  INFO bore_cli::client: listening at bore.pub:61081
2026-07-06T08:52:32.878942Z  INFO proxy{id=0c27ab2a-a69d-4a09-b373-4f804f518955}: bore_cli::client: new connection
2026-07-06T08:52:33.179968Z  INFO proxy{id=aabc274c-ae52-4cb7-b7f3-4a708984f9c8}: bore_cli::client: new connection
2026-07-06T08:52:35.028680Z  INFO proxy{id=872bc27d-5c81-4f0d-a839-ada8ff2aa364}: bore_cli::client: new connection
2026-07-06T08:52:39.709330Z  INFO proxy{id=0c27ab2a-a69d-4a09-b373-4f804f518955}: bore_cli::client: connection exited
2026-07-06T08:52:39.715538Z  INFO proxy{id=aabc274c-ae52-4cb7-b7f3-4a708984f9c8}: bore_cli::client: connection exited
2026-07-06T08:52:40.086918Z  INFO proxy{id=872bc27d-5c81-4f0d-a839-ada8ff2aa364}: bore_cli::client: connection exited
2026-07-0